# Occlusion-aware fingerprints on three ReID models (MEVID-SC)

**Question:** is our fix a plug-in that helps *any* ReID model, not just CSCI?

| Model | Type | Weights |
|---|---|---|
| CSCI | Large video transformer, state of the art on MEVID | Already run (main and controls) |
| FastReID SBS R101-IBN | Strong image ReID network | MSMT17, from the FastReID GitHub release |
| OSNet x1_0 | Lightweight image ReID model | MSMT17 |

**For each model, four fingerprints per clip.** The controls remove the **same number** of frames per clip as ours:
1. all frames;
2. **ours:** only frames where the pose model sees at least half the body;
3. control: random frames removed;
4. control: the frames that look farthest from the clip average removed.

**Notes:**
- The image models use every 4th frame, which are the frames the pose model checked.
- No labels, no training, all models unchanged.
- Same fair test as before: the same 25 random splits of people, both directions.
- **Check:** the CSCI rows should reproduce the earlier numbers (IDF1 81.3 → 85.6).

**Needs:** an A100 and the earlier OcclusionAware results on Drive. About 30 to 40 minutes. Run cells 1 to 5.

In [ ]:
# 1. Setup: Drive, paths, frozen settings, MEVID-SC test frames (local).
import os, glob, json, time, subprocess, sys, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(pats[0])

SC_DIR = os.path.dirname(find([f'{SD}/Data/MEVID-SameClothes/SUBSET_REPORT.json',
                               f'{SD}/**/MEVID-SameClothes/SUBSET_REPORT.json']))
OA = os.path.dirname(find([f'{SD}/Data/MEVID-results/OcclusionAware-SameClothes/'
                           'filter_visible.npz',
                           f'{SD}/**/OcclusionAware-SameClothes/filter_visible.npz']))
CS = os.path.join(os.path.dirname(OA), 'CSCI-SameClothes')
OUT = os.path.join(os.path.dirname(OA), 'ThreeModels-SameClothes')
os.makedirs(OUT, exist_ok=True)
SETTINGS = {
    'models': ['CSCI (video transformer, state of the art)',
               'FastReID SBS R101-IBN (MSMT17)',
               'OSNet x1_0 (MSMT17)'],
    'image_models_use': 'every 4th frame (the frames the pose model '
                        'checked); clip fingerprint = mean of L2-normalised '
                        'frame features',
    'ours': 'keep frames where the person is found with >= 50% of body '
            'keypoints visible (the frozen main variant); clips with too '
            'few good frames unchanged',
    'controls': 'random / feature-distance: remove the SAME number of '
                'frames per clip as ours',
    'precision_target': 0.90, 'n_splits': 25, 'seed': 2027,
    'control_seed': 2027, 'check_every': 4, 'visible_thr': 0.5,
    'min_kept_frames': 8}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
st = SETTINGS

A = f'{SC_DIR}/annotations'
names = [l.strip() for l in open(f'{A}/test_name.txt') if l.strip()]
info = np.loadtxt(f'{A}/track_test_info.txt').astype(np.int64)
qidx = np.loadtxt(f'{A}/query_IDX.txt').astype(np.int64)
gidx = np.setdiff1d(np.arange(len(info)), qidx)
pid, cam = info[:, 2], info[:, 4]
LOCAL = '/content/sc_test'
if not os.path.isdir(f'{LOCAL}/bbox_test'):
    os.makedirs(LOCAL, exist_ok=True)
    t0 = time.time(); print('Unpacking the MEVID-SC test frames ...')
    subprocess.run(['tar', '-xf', f'{SC_DIR}/mevid-sc-bbox-test.tar', '-C',
                    LOCAL], check=True)
    print(f'done in {time.time() - t0:.0f} s')
print(f'{len(info)} clips, {len(np.unique(pid))} people | results -> {OUT}')


In [ ]:
# 2. The checked frames and OUR keep-mask on them (from the saved pose
#    checks), plus the number of frames each control must remove.
check = []
for s, e in info[:, :2]:
    c = list(range(s, e + 1, st['check_every']))
    if c[-1] != e:
        c.append(e)
    check += c
check = np.array(check)
parts = [np.load(f) for f in sorted(glob.glob(f'{OA}/personcheck_*.npz'))]
K = np.concatenate([p['k'] for p in parts])
assert len(K) == len(check), 'pose checks missing in OcclusionAware folder'
FOUND = np.concatenate([p['found'] for p in parts])
VIS = np.concatenate([p['vis'] for p in parts])
o = np.argsort(K); K, FOUND, VIS = K[o], FOUND[o], VIS[o]
RANGES, KEEP, KEPT_ALL = [], np.ones(len(K), bool), []
for t, (s, e, *_) in enumerate(info):
    lo, hi = np.searchsorted(K, s), np.searchsorted(K, e, 'right')
    RANGES.append((lo, hi))
    fr = np.arange(s, e + 1)
    near = lo + np.abs(fr[:, None] - K[lo:hi][None, :]).argmin(1)
    keep_all = (FOUND[near] == 1) & (VIS[near] >= st['visible_thr'])
    if keep_all.sum() >= st['min_kept_frames']:     # same rule as main run
        KEPT_ALL.append(keep_all.sum())
        KEEP[lo:hi] = (FOUND[lo:hi] == 1) & (VIS[lo:hi] >= st['visible_thr'])
        if not KEEP[lo:hi].any():                   # never leave a clip empty
            KEEP[lo:hi] = True
    else:                                           # too few: clip unchanged
        KEPT_ALL.append(len(fr))
assert (np.array(KEPT_ALL) == np.load(f'{OA}/filter_visible.npz')['kept']).all(), \
    'rule differs from the main run'
REMOVE = np.array([(~KEEP[lo:hi]).sum() for lo, hi in RANGES])
print(f'{len(K):,} checked frames; ours removes {(~KEEP).sum():,} '
      f'({100 * (~KEEP).mean():.1f}%) in {(REMOVE > 0).sum()} clips '
      '(matches the main CSCI run)')


In [ ]:
# 3. Per-frame features for the two image models on the checked frames
#    (about 240,000 images; ~5 min OSNet, ~10 min FastReID on an A100).
#    Kept on local disk only (large); clip fingerprints go to Drive.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchreid',
                'gdown', 'yacs', 'termcolor', 'tabulate'], check=True)
import torch, torch.nn.functional as F
import torchvision.transforms as TV
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


class Frames(Dataset):
    def __init__(self, tf): self.tf = tf
    def __len__(self): return len(K)
    def __getitem__(self, i):
        n = names[K[i]]
        return self.tf(Image.open(f'{LOCAL}/bbox_test/{n[:4]}/{n}').convert('RGB'))


def run(model, tf, dim, path, bs=256):
    if os.path.exists(path):
        return np.load(path, mmap_mode='r')
    out = np.lib.format.open_memmap(path + '.tmp', 'w+', np.float16, (len(K), dim))
    i = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for x in tqdm(DataLoader(Frames(tf), batch_size=bs, num_workers=8,
                                 pin_memory=True), desc=os.path.basename(path)):
            f = model(x.cuda(non_blocking=True))
            f = f[0] if isinstance(f, (tuple, list)) else f
            f = F.normalize(f.float(), dim=1)
            out[i:i + len(f)] = f.cpu().numpy(); i += len(f)
    out.flush(); del out
    os.rename(path + '.tmp', path)
    return np.load(path, mmap_mode='r')


FRAME = {}
# --- OSNet x1_0, MSMT17 (lightweight image ReID model) ---
import torchreid, gdown
m = torchreid.models.build_model(name='osnet_x1_0', num_classes=1000,
                                 loss='softmax', pretrained=True)
ck = os.path.expanduser('~/.cache/torch/checkpoints/osnet_x1_0_msmt17.pth')
os.makedirs(os.path.dirname(ck), exist_ok=True)
for url in ['https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M',
            'https://drive.google.com/uc?id=1IosIFlLiulGIjwW3H8uMRmx3MzPwf86x']:
    if not os.path.exists(ck):
        try:
            gdown.download(url, ck, quiet=True, fuzzy=True)
        except Exception as ex:
            print('download failed:', ex)
torchreid.utils.load_pretrained_weights(m, ck)
tf_os = TV.Compose([TV.Resize((256, 128)), TV.ToTensor(),
                    TV.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
FRAME['OSNet x1_0 (MSMT17)'] = run(
    m.cuda().eval(), tf_os, 512, '/content/frames_osnet.npy', 512)
del m; torch.cuda.empty_cache()

# --- FastReID SBS R101-IBN, MSMT17 (weights from the FastReID release) ---
FR = '/content/fast-reid'
if not os.path.isdir(FR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/JDAI-CV/'
                    'fast-reid.git', FR], check=True)
    for f in [f'{FR}/fastreid/data/build.py', f'{FR}/fastreid/evaluation/testing.py']:
        s = open(f).read()
        s = s.replace('from collections import Mapping, OrderedDict',
                      'from collections import OrderedDict\n'
                      'from collections.abc import Mapping')
        s = s.replace('from collections import Mapping',
                      'from collections.abc import Mapping')
        open(f, 'w').write(s)
WFR = '/content/msmt_sbs_R101-ibn.pth'
if not os.path.exists(WFR):
    subprocess.run(['wget', '-q', '-O', WFR, 'https://github.com/JDAI-CV/'
                    'fast-reid/releases/download/v0.1.1/msmt_sbs_R101-ibn.pth'],
                   check=True)
sys.path.insert(0, FR)
from fastreid.config import get_cfg
from fastreid.modeling.meta_arch import build_model
from fastreid.utils.checkpoint import Checkpointer
cfg = get_cfg()
cfg.merge_from_file(f'{FR}/configs/MSMT17/sbs_R101-ibn.yml')
cfg.MODEL.BACKBONE.PRETRAIN = False
cfg.freeze()
m = build_model(cfg).cuda().eval()
Checkpointer(m).load(WFR)
tf_fr = TV.Compose([TV.Resize((384, 128)), TV.PILToTensor(),
                    TV.Lambda(lambda t: t.float())])   # 0-255 RGB; model normalises
FRAME['FastReID SBS R101-IBN (MSMT17)'] = run(
    m, tf_fr, 2048, '/content/frames_fastreid.npy', 256)
del m; torch.cuda.empty_cache()
print({k: v.shape for k, v in FRAME.items()})


In [ ]:
# 4. Clip fingerprints for every model and variant. Image models: mean of
#    the chosen frames. CSCI: the fingerprints already computed (original,
#    ours, random, feature-distance) in the OcclusionAware folder.
def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)


VARIANTS = ['all frames', 'ours (pose: person visible)',
            'control: random frames', 'control: feature-distance']
FEATS = {}
rng = np.random.default_rng(st['control_seed'])
for mname, Fm in FRAME.items():
    Fm = np.asarray(Fm, dtype=np.float32)
    out = {v: np.zeros((len(info), Fm.shape[1]), np.float32) for v in VARIANTS}
    for t, (lo, hi) in enumerate(RANGES):
        f = Fm[lo:hi]; k = REMOVE[t]
        out['all frames'][t] = f.mean(0)
        out['ours (pose: person visible)'][t] = f[KEEP[lo:hi]].mean(0)
        r = np.ones(hi - lo, bool)
        if k:
            r[rng.choice(hi - lo, k, replace=False)] = False
        out['control: random frames'][t] = f[r].mean(0)
        mu = unit(f.mean(0, keepdims=True))[0]
        d = np.ones(hi - lo, bool)
        if k:
            d[np.argsort(-(1 - f @ mu), kind='stable')[:k]] = False
        out['control: feature-distance'][t] = f[d].mean(0)
    for v in VARIANTS:
        FEATS[(mname, v)] = unit(out[v])
    np.savez(f'{OUT}/fingerprints_{mname.split()[0]}.npz',
             **{v: out[v] for v in VARIANTS}, pid=pid, cam=cam)
csci_files = {'all frames': f'{CS}/test_templates_csci_sc.npz',
              'ours (pose: person visible)': f'{OA}/test_templates_csci_visible.npz',
              'control: random frames': f'{OA}/test_templates_csci_random.npz',
              'control: feature-distance': f'{OA}/test_templates_csci_feature.npz'}
for v, fn in csci_files.items():
    z = np.load(fn); assert (z['pid'] == pid).all()
    FEATS[('CSCI (video transformer, state of the art)', v)] = unit(z['templates'])
print(len(FEATS), 'fingerprint sets ready')


In [ ]:
# 5. Results for all three models: accuracy and consistent ID (the cut
#    calibrated on one group of people, tested on the other; the SAME 25
#    random splits x 2 directions as the CSCI runs; every clip counts).
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon


def ap_cmc(order, good, junk):
    order = order[~np.isin(order, junk)]
    hits = np.isin(order, good)
    cmc = np.zeros(len(order)); cmc[np.argmax(hits):] = 1.0
    r = np.flatnonzero(hits)
    return np.mean((np.arange(len(r)) + 1) / (r + 1)), cmc


def accuracy(X):
    S = X[qidx] @ X[gidx].T
    aps, r1 = [], []
    for i, q in enumerate(qidx):
        sp = np.flatnonzero(pid[gidx] == pid[q])
        sc = np.flatnonzero(cam[gidx] == cam[q])
        good, junk = np.setdiff1d(sp, sc), np.intersect1d(sp, sc)
        if good.size:
            a, c = ap_cmc(np.argsort(-S[i]), good, junk)
            aps.append(a); r1.append(c[0])
    return {'mAP %': 100 * np.mean(aps), 'Rank-1 %': 100 * np.mean(r1)}


def score(pred, p, c):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1)); np.add.at(M, (pi, ci), 1)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0); tp, same = (M * (M - 1)).sum(), (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    Kc = np.zeros((M.shape[0], ki.max() + 1, M.shape[1]))
    np.add.at(Kc, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (Kc.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (Kc ** 2).sum((1, 2))).sum()
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'pair precision %': 100 * tp / same if same else 100.0,
            'cross-camera recall %': 100 * x_same / max(1, x_tot)}


def tree(X, idx):
    D = np.clip(1 - X[idx] @ X[idx].T, 0, None); D = (D + D.T) / 2
    np.fill_diagonal(D, 0)
    return linkage(squareform(D, checks=False), 'average')


def calibrate(L, idx):
    best = 0.01
    for t in np.linspace(0.01, 1.5, 150):
        if score(fcluster(L, t, 'distance'), pid[idx], cam[idx]
                 )['pair precision %'] >= 100 * st['precision_target']:
            best = t
        else:
            break
    return best


people = np.unique(pid)
rng = np.random.default_rng(st['seed'])
SPLITS = []
for s in range(st['n_splits']):
    perm = rng.permutation(people)
    a = np.isin(pid, perm[:len(people) // 2])
    SPLITS.append([np.flatnonzero(a), np.flatnonzero(~a)])
ROWS = []
for (mname, v), X in tqdm(FEATS.items(), desc='models x variants'):
    acc = accuracy(X)
    for s, G in enumerate(SPLITS):
        Ls = [tree(X, g) for g in G]
        for c, t in [(0, 1), (1, 0)]:
            r = score(fcluster(Ls[t], calibrate(Ls[c], G[c]), 'distance'),
                      pid[G[t]], cam[G[t]])
            r.update(model=mname, variant=v, split=s, direction=c, **acc)
            ROWS.append(r)
REP = pd.DataFrame(ROWS)
REP.to_csv(f'{OUT}/all_models_50tests.csv', index=False)
COLS = ['IDF1 %', 'IDs per person', 'cross-camera recall %',
        'pair precision %', 'mAP %', 'Rank-1 %']
MAIN = REP.groupby(['model', 'variant'], sort=False)[COLS].mean().round(2)
print('Mean over 50 tests (mAP / Rank-1: one fixed query set):')
display(MAIN)
MAIN.to_csv(f'{OUT}/all_models_summary.csv')

PAIR = []
for mname in st['models']:
    R = REP[REP.model == mname]
    ours = R[R.variant == 'ours (pose: person visible)'].set_index(['split', 'direction'])
    for ref in ['all frames', 'control: random frames', 'control: feature-distance']:
        b = R[R.variant == ref].set_index(['split', 'direction'])
        for metric, better in [('IDF1 %', 1), ('IDs per person', -1),
                               ('cross-camera recall %', 1)]:
            d = ours[metric] - b[metric]
            PAIR.append({'model': mname.split(' (')[0], 'ours vs': ref,
                         'metric': metric, 'mean change': round(d.mean(), 2),
                         'ours better in % of tests': round(100 * (better * d > 0).mean()),
                         'p (Wilcoxon)': f'{wilcoxon(d).pvalue:.1e}' if (d != 0).any() else 'n/a'})
PAIR = pd.DataFrame(PAIR)
PAIR.to_csv(f'{OUT}/all_models_paired.csv', index=False)
print('\nOurs vs original and vs the two controls, per model (50 tests):')
display(PAIR)
import zipfile
with zipfile.ZipFile('/content/ThreeModels_results.zip', 'w') as z:
    for f in glob.glob(f'{OUT}/*.csv') + glob.glob(f'{OUT}/*.json'):
        z.write(f, os.path.basename(f))
print('Saved to', OUT, '\nZip: /content/ThreeModels_results.zip')


**Cell 6 (standalone)** rescoring all saved fingerprints with the fine-grid calibrator used throughout the paper; it writes `final_*.csv` (paper Table 2).

In [ ]:
# 6. STANDALONE final results (run after a restart too; needs only Drive).
#    Rescores the saved fingerprints of all three models x four variants with
#    the fine-grid calibrator used everywhere in the paper (Table 2).
import os, glob, zipfile, subprocess, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
from tqdm.auto import tqdm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon
drive.mount('/content/drive')
SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(pats[0])
SC_DIR = os.path.dirname(find([f'{SD}/Data/MEVID-SameClothes/SUBSET_REPORT.json',
                               f'{SD}/**/MEVID-SameClothes/SUBSET_REPORT.json']))
OA = os.path.dirname(find([f'{SD}/Data/MEVID-results/OcclusionAware-SameClothes/filter_visible.npz',
                           f'{SD}/**/OcclusionAware-SameClothes/filter_visible.npz']))
CS = os.path.join(os.path.dirname(OA), 'CSCI-SameClothes')
OUT = os.path.join(os.path.dirname(OA), 'ThreeModels-SameClothes')
A = f'{SC_DIR}/annotations'
info = np.loadtxt(f'{A}/track_test_info.txt').astype(np.int64)
qidx = np.loadtxt(f'{A}/query_IDX.txt').astype(np.int64)
gidx = np.setdiff1d(np.arange(len(info)), qidx)
pid, cam = info[:, 2], info[:, 4]
z0 = np.load(f'{CS}/test_templates_csci_sc.npz'); assert (z0['pid'] == pid).all()
st = {'precision_target': 0.90, 'n_splits': 25, 'seed': 2027}

def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)

V = ['all frames', 'ours (pose: person visible)',
     'control: random frames', 'control: feature-distance']
MODELS = ['CSCI (video transformer, state of the art)',
          'FastReID SBS R101-IBN (MSMT17)', 'OSNet x1_0 (MSMT17)']
FEATS = {}
for v, fn in zip(V, [None, 'visible', 'random', 'feature']):
    if fn is None:
        FEATS[(MODELS[0], v)] = unit(z0['templates'])
    elif os.path.exists(f'{OA}/test_templates_csci_{fn}.npz'):
        FEATS[(MODELS[0], v)] = unit(np.load(f'{OA}/test_templates_csci_{fn}.npz')['templates'])
    else:
        print('not run yet: CSCI', v)
for mname, short in zip(MODELS[1:], ['FastReID', 'OSNet']):
    fn = f'{OUT}/fingerprints_{short}.npz'
    if os.path.exists(fn):
        z = np.load(fn); assert (z['pid'] == pid).all()
        for v in V:
            FEATS[(mname, v)] = unit(z[v])
    else:
        print('not run yet:', mname)

def ap_cmc(order, good, junk):
    order = order[~np.isin(order, junk)]
    hits = np.isin(order, good)
    cmc = np.zeros(len(order)); cmc[np.argmax(hits):] = 1.0
    r = np.flatnonzero(hits)
    return np.mean((np.arange(len(r)) + 1) / (r + 1)), cmc

def accuracy(X):
    S = X[qidx] @ X[gidx].T
    aps, r1 = [], []
    for i, q in enumerate(qidx):
        sp = np.flatnonzero(pid[gidx] == pid[q])
        sc = np.flatnonzero(cam[gidx] == cam[q])
        good, junk = np.setdiff1d(sp, sc), np.intersect1d(sp, sc)
        if good.size:
            a, c = ap_cmc(np.argsort(-S[i]), good, junk)
            aps.append(a); r1.append(c[0])
    return {'mAP %': 100 * np.mean(aps), 'Rank-1 %': 100 * np.mean(r1)}

def score(pred, p, c):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1)); np.add.at(M, (pi, ci), 1)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0); tp, same = (M * (M - 1)).sum(), (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    Kc = np.zeros((M.shape[0], ki.max() + 1, M.shape[1])); np.add.at(Kc, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (Kc.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (Kc ** 2).sum((1, 2))).sum()
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'pair precision %': 100 * tp / same if same else 100.0,
            'cross-camera recall %': 100 * x_same / max(1, x_tot)}

def tree(X, idx):
    D = np.clip(1 - X[idx] @ X[idx].T, 0, None); D = (D + D.T) / 2
    np.fill_diagonal(D, 0)
    return linkage(squareform(D, checks=False), 'average')

CUTS = np.geomspace(0.001, 1.5, 120)
def calibrate(L, idx):
    best = CUTS[0]
    for t in CUTS:
        if score(fcluster(L, t, 'distance'), pid[idx], cam[idx]
                 )['pair precision %'] >= 100 * st['precision_target']:
            best = t
    return best

people = np.unique(pid)
rng = np.random.default_rng(st['seed'])
SPLITS = []
for s in range(st['n_splits']):
    perm = rng.permutation(people)
    a = np.isin(pid, perm[:len(people) // 2])
    SPLITS.append([np.flatnonzero(a), np.flatnonzero(~a)])
ROWS = []
for (mname, v), X in tqdm(FEATS.items(), desc='models x variants'):
    acc = accuracy(X)
    for s, G in enumerate(SPLITS):
        Ls = [tree(X, g) for g in G]
        for c, t in [(0, 1), (1, 0)]:
            r = score(fcluster(Ls[t], calibrate(Ls[c], G[c]), 'distance'),
                      pid[G[t]], cam[G[t]])
            r.update(model=mname, variant=v, split=s, direction=c, **acc)
            ROWS.append(r)
REP = pd.DataFrame(ROWS)
REP.to_csv(f'{OUT}/final_all_models_50tests.csv', index=False)
COLS = ['IDF1 %', 'IDs per person', 'cross-camera recall %',
        'pair precision %', 'mAP %', 'Rank-1 %']
MAIN = REP.groupby(['model', 'variant'], sort=False)[COLS].mean().round(2)
print('MEVID-SC, mean over 50 tests:'); display(MAIN)
MAIN.to_csv(f'{OUT}/final_all_models_summary.csv')
PAIR = []
for mname in MODELS:
    R = REP[REP.model == mname]
    if (R.variant == V[1]).sum() == 0:
        continue
    ours = R[R.variant == V[1]].set_index(['split', 'direction'])
    for ref in [V[0], V[2], V[3]]:
        if (R.variant == ref).sum() == 0:
            continue
        b = R[R.variant == ref].set_index(['split', 'direction'])
        for metric, better in [('IDF1 %', 1), ('IDs per person', -1),
                               ('cross-camera recall %', 1)]:
            d = ours[metric] - b[metric]
            PAIR.append({'model': mname.split(' (')[0], 'ours vs': ref,
                         'metric': metric, 'mean change': round(d.mean(), 2),
                         'ours better in % of tests': round(100 * (better * d > 0).mean()),
                         'p (Wilcoxon)': f'{wilcoxon(d).pvalue:.1e}' if (d != 0).any() else 'n/a'})
PAIR = pd.DataFrame(PAIR)
PAIR.to_csv(f'{OUT}/final_all_models_paired.csv', index=False)
print('\nOurs vs original and controls (50 tests):'); display(PAIR)
with zipfile.ZipFile('/content/ThreeModels_final_results.zip', 'w') as zf:
    for f in glob.glob(f'{OUT}/final_*.csv') + glob.glob(f'{OUT}/*.json'):
        zf.write(f, os.path.basename(f))
print('Saved to', OUT, '\nZip: /content/ThreeModels_final_results.zip')
